## Create client

In [ ]:
from scray.job_client.config import ScrayJobClientConfig
from scray.job_client.client import ScrayJobClient 

config = ScrayJobClientConfig(
  host_address = "http://ml-integration.research.dev.seeburger.de",
  #host_address = "http://localhost",
  port = 8082
)

client = ScrayJobClient(config=config)

In [17]:
env = "http://scray.org/ai/jobs/env/see/ticket-project/provence-data/prod/ticket_update"
 
# Blocking till new job apears
#jobs = client.get_jobs(processing_env=env, requested_state="FINISHED") 

#print(len(jobs))
#for job_name in jobs:
#    print("Process uploaded data from job: " + job_name)

In [ ]:
import time

jobs1_times = []
jobs2_times = []

for i in range(1, 11):
    print(f"\nRun {i}:")

    start = time.time()
    jobs1 = client.get_jobs(processing_env=env)
    end = time.time()
    t1 = end - start
    jobs1_times.append(t1)
    print(f"Fetching jobs1 consumed time = {t1:.4f} seconds, count = {len(jobs1)}")

    start = time.time()
    jobs2 = client.get_jobs(processing_env=env, requested_state="FINISHED")
    end = time.time()
    t2 = end - start
    jobs2_times.append(t2)
    print(f"Fetching jobs2 consumed time = {t2:.4f} seconds, count = {len(jobs2)}")

# --- Summary statistics ---
def summarize(times, label):
    avg = sum(times) / len(times)
    print(
        f"\n{label} statistics over {len(times)} runs:\n"
        f"  Average: {avg:.4f} seconds\n"
        f"  Min:     {min(times):.4f} seconds\n"
        f"  Max:     {max(times):.4f} seconds"
    )

summarize(jobs1_times, "Client filterd fetch")
summarize(jobs2_times, "Indexed state fetch")


In [18]:
from collections import Counter

def countStates(state):
    jobs_new = client.get_jobs(processing_env=env, requested_state=state)
    jobs_old = client.get_jobs_old(processing_env=env, requested_state=state)

    print("---------------------------------------------------------------------------")
    print(f"State: {state}")
    print(f"\tNew (get_jobs): {len(jobs_new)}")
    print(f"\tOld (get_jobs_old): {len(jobs_old)}")

    # Differences
    only_in_new = set(jobs_new) - set(jobs_old)
    only_in_old = set(jobs_old) - set(jobs_new)

    if only_in_new:
        print("\nJobs only in get_jobs:")
        for job in sorted(only_in_new):
            print(f"  - {job}")

    if only_in_old:
        print("\nJobs only in get_jobs_old:")
        for job in sorted(only_in_old):
            print(f"  - {job}")

    if not only_in_new and not only_in_old:
        print("\n✅ Both lists contain the same jobs.")

    print("---------------------------------------------------------------------------")

    # return both lists if needed
    return jobs_new, jobs_old


In [22]:
countStates("CERTIFICATE_TO_MFT/INSTALL")
countStates("UPDATED")
countStates("FINISHED")
countStates("PUBLISHED")

---------------------------------------------------------------------------
State: CERTIFICATE_TO_MFT/INSTALL
	New (get_jobs): 1
	Old (get_jobs_old): 1

✅ Both lists contain the same jobs.
---------------------------------------------------------------------------
---------------------------------------------------------------------------
State: UPDATED
	New (get_jobs): 967
	Old (get_jobs_old): 967

✅ Both lists contain the same jobs.
---------------------------------------------------------------------------


Error while interacting with sync API (POST). Code: 404, Response: No entry found for given filter


---------------------------------------------------------------------------
State: FINISHED
	New (get_jobs): 75425
	Old (get_jobs_old): 75425

✅ Both lists contain the same jobs.
---------------------------------------------------------------------------
---------------------------------------------------------------------------
State: PUBLISHED
	New (get_jobs): 0
	Old (get_jobs_old): 0

✅ Both lists contain the same jobs.
---------------------------------------------------------------------------


([], [])

In [ ]:
import time


jobs2_times = []

env = "http://scray.org/ai/jobs/env/see/ticket-project/provence-data/prod/ticket_update"

def print_num_states(state):
    jobs2 = client.get_jobs(processing_env=env, requested_state=state)
    print("State: " + state)
    print("\tNum jobs " + str(len(jobs2)))
    print("\tNames " + str(jobs2))

print_num_states("CERTIFICATE_TO_MFT/INSTALL")
print_num_states("UPDATED")
print_num_states("FINISHED")
print_num_states("PUBLISHED")